# 10 — Final Test Evaluation and Error Analysis

## Goal
Evaluate the selected model on the held-out test data and inspect mistakes.

**What happens using this code?** It loads the model selected using validation data, computes final accuracy and macro-F1, writes predictions, and lists misclassified questions. Run this after notebook 09.

In [ ]:
from pathlib import Path
import json, joblib, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "datasets/processed"
test_df = pd.read_csv(DATA / "paddy_test.csv", keep_default_na=False, encoding="utf-8-sig")
selection_path = ROOT / "models/model_evaluation_results/selected_model.json"
selection = json.loads(selection_path.read_text(encoding="utf-8"))
model_name = selection["selected_model"]

if model_name == "tfidf_linear_svc":
    model = joblib.load(ROOT / "models/problem_classifier/tfidf_linear_svc.joblib")
    pred = model.predict(test_df["question"])
elif model_name == "xlm_roberta_base":
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
    model_dir = ROOT / "models/problem_classifier/xlm_roberta_base/best_model"
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    infer_ds = Dataset.from_pandas(test_df[["question"]], preserve_index=False).map(lambda x: tokenizer(x["question"], truncation=True, max_length=160), batched=True)
    trainer = Trainer(model=model, args=TrainingArguments(output_dir=str(ROOT/"models/tmp_test_eval"), per_device_eval_batch_size=8, report_to="none"), data_collator=DataCollatorWithPadding(tokenizer), processing_class=tokenizer)
    output = trainer.predict(infer_ds)
    id2label = {int(k):v for k,v in model.config.id2label.items()}
    pred = [id2label[int(i)] for i in np.argmax(output.predictions, axis=1)]
else:
    raise ValueError(f"Unknown selected model: {model_name}")

p, r, f1, _ = precision_recall_fscore_support(test_df["label"], pred, average="macro", zero_division=0)
metrics = {"selected_model":model_name, "accuracy":float(accuracy_score(test_df["label"], pred)), "precision_macro":float(p), "recall_macro":float(r), "f1_macro":float(f1)}
print(json.dumps(metrics, indent=2))
print(classification_report(test_df["label"], pred, zero_division=0))

predictions = test_df[["qa_id", "pair_id", "question", "language", "label", "issue_id"]].copy()
predictions["predicted_label"] = pred
predictions["correct"] = predictions["label"] == predictions["predicted_label"]
result_dir = ROOT / "models/model_evaluation_results"
predictions.to_csv(result_dir / "final_test_predictions.csv", index=False, encoding="utf-8-sig")
(result_dir / "final_test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("\nMisclassified examples:")
display(predictions[~predictions["correct"]].head(25))
print("\nAccuracy by language:")
print(predictions.groupby("language").apply(lambda x: (x.label == x.predicted_label).mean(), include_groups=False))

**Reporting note:** report macro-F1, per-category precision/recall, and Sinhala-versus-English results alongside accuracy. Since this dataset is synthetic and not expert-reviewed, do not present the result as proven real-world farmer accuracy.